# Lab 63: OCR-reading on real rendered images

Run a real OCR engine on real images across a degradation ladder, and grade reads by CER and a value-aware answer check. Fill in the `TODO` cell; reference in `solution/`. Concept: [concepts/rag/ocr-on-real-images.md](../../concepts/rag/ocr-on-real-images.md).

## Step 0: Setup

In [ ]:
from images import CORPUS, build_image
from read import grade, tesseract_available, EXPECTED_OCR, answer_read_ok, numbers_in
# A real OCR engine (tesseract) reads real rendered images. We render a deterministic corpus across
# a degradation ladder, OCR it, and grade two ways: CER (legibility) and a value-aware answer check.
print("tesseract available:", tesseract_available())
print(f"{len(CORPUS)} images:", ", ".join(f"{cid}[{deg}]" for cid,_,_,deg in CORPUS))

## Step 1: Grade the real reads

In [ ]:
# TODO: grade(live=tesseract_available()) and inspect the per-row table. Which images read
# perfectly, which fail to read at all, and which read text but the wrong number? Why is the
# "9.4 billion" -> "94 bien" misread a numeric failure rather than a legibility one?
raise NotImplementedError

## Step 2: The real numeric misread (9.4 billion -> 94)

In [ ]:
# The dangerous case, for real: heavy degradation makes tesseract read "9.4 billion" as "94 bien".
# CER is only moderate, but the value is off by ~10^8 - caught only by the value-aware check.
print("OCR of the heavy image:", EXPECTED_OCR["hvy"])
print("numbers parsed from it:", numbers_in(EXPECTED_OCR["hvy"]))
print("answer '9.4 billion' read correctly?", answer_read_ok("9.4 billion", EXPECTED_OCR["hvy"]))

## Step 3: The standalone-number guard

In [ ]:
# The standalone-number guard matters: 'Q4 revenue 4.2 million' must read the value as 4.2M, not 4.
print(numbers_in("Q4 revenue 4.2 million"), "->", answer_read_ok("4.2 million", "Q4 revenue 4.2 million"))

## What you built

A real OCR-reading evaluation: real pixels, a real engine, real failures. `images.py` renders a deterministic corpus across a degradation ladder (clean, sensor noise, blur, small-font blur, heavy blur+noise); `read.py` runs tesseract and grades each read by CER (legibility) and a value-aware answer check (correctness), attributing every wrong read to a stage. On this corpus the engine reads clean and mildly-blurred images perfectly, returns a blank on heavy sensor noise (a read failure), garbles a small blurred line to 'hooey', and - the case that matters - misreads '9.4 billion' as '94 bien', a ~10^8 value error that CER alone (0.27) would wave through.

**Where this is real, and where it simplifies:** the images and the OCR are real; the corpus is small and the 'images' are cropped text regions rather than full charts, which is what a layout detector would feed an OCR engine anyway. Because tesseract output varies by version, the module ships a recorded fixture so the grading is deterministic everywhere and `--live` runs the real engine when it's installed. The lesson from Labs 61-62 now holds on real data: score legibility and value separately, because a real misread of a single digit is a small CER and a catastrophic answer. Math: [math-foundations/18](../../math-foundations/18-edit-distance-alignment.md); concept: [concepts/rag/ocr-on-real-images.md](../../concepts/rag/ocr-on-real-images.md).